In [33]:
from pathlib import Path
import re
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

import joblib
import sklearn


from flask import Flask, request, jsonify

# Data Collection and Preprocessing

### - Locate and check the dataset

In [14]:
# SETUP: Locate the Sentiment140 CSV file in the project folder.
input_file = Path.cwd() / "training.1600000.processed.noemoticon.csv"
# Set the folder and filename for the processed data
output_dir = Path.cwd() / "data" / "processed"
output_file = output_dir / "cleaned_tweets.csv"

# Check that the notebook can find the dataset before loading it.
print(input_file)
print(input_file.exists())

C:\Users\steph\Desktop\Willis College Courses\Fundamental of Predictive Analysis ML\Final Project\training.1600000.processed.noemoticon.csv
True


### - Load and inspect the tweet

In [6]:
# Column names
columns = ["sentiment", "tweet_id", "date", "query", "user", "text"]
# Load the dataset using its original text encoding
df = pd.read_csv(input_file, encoding="latin-1", header=None, names=columns)

#  Check the size and available sentiment labels
print("Dataset shape:", df.shape)
print(df[["sentiment", "text"]].head())

print("\nSentiment label counts:")
print(df["sentiment"].value_counts())

Dataset shape: (1600000, 6)
   sentiment                                               text
0          0  @switchfoot http://twitpic.com/2y1zl - Awww, t...
1          0  is upset that he can't update his Facebook by ...
2          0  @Kenichan I dived many times for the ball. Man...
3          0    my whole body feels itchy and like its on fire 
4          0  @nationwideclass no, it's not behaving at all....

Sentiment label counts:
sentiment
0    800000
4    800000
Name: count, dtype: int64


### - Remove invalid rows and duplicates

In [7]:
# Remove rows that have no tweet text or no sentiment label
df = df.dropna(subset=["text", "sentiment"]).copy()

In [8]:
# Remove duplicate tweets
df = df.drop_duplicates(subset=["text"]).copy()

print("Rows remaining after removing invalid and duplicate tweets:", len(df))

Rows remaining after removing invalid and duplicate tweets: 1581466


### - Clean, lowercase, and tokenize the tweets

In [9]:
# Clean one tweet by removing URLs, mentions, and special characters
def clean_and_tokenize(tweet):
    # Convert the tweet to lowercase
    tweet = str(tweet).lower()
    # Remove web links and user mentions
    tweet = re.sub(r"https?://\S+|www\.\S+", " ", tweet)
    tweet = re.sub(r"@\w+", " ", tweet)
    # Replace punctuation, numbers, and other special characters with spaces
    tweet = re.sub(r"[^a-z\s]", " ", tweet)
    # Split the cleaned text into lowercase word tokens
    tokens = tweet.split()
    # Join the tokens so they can be saved in a CSV column
    return " ".join(tokens)

In [10]:
# Apply the cleaning and tokenization to every tweet
df["clean_text"] = df["text"].map(clean_and_tokenize)

In [11]:
# Remove tweets that became empty after cleaning
df = df[df["clean_text"].str.strip() != ""].copy()

In [12]:
# Preview the original and cleaned tweet text
print(df[["text", "clean_text"]].head())

                                                text  \
0  @switchfoot http://twitpic.com/2y1zl - Awww, t...   
1  is upset that he can't update his Facebook by ...   
2  @Kenichan I dived many times for the ball. Man...   
3    my whole body feels itchy and like its on fire    
4  @nationwideclass no, it's not behaving at all....   

                                          clean_text  
0  awww that s a bummer you shoulda got david car...  
1  is upset that he can t update his facebook by ...  
2  i dived many times for the ball managed to sav...  
3     my whole body feels itchy and like its on fire  
4  no it s not behaving at all i m mad why am i h...  


### - Save the processed data

In [15]:
# Create the output folder if it does not already exist
output_dir.mkdir(parents=True, exist_ok=True)

In [16]:
# Save the sentiment label and cleaned tweet text
processed_df = df[["sentiment", "clean_text"]]
processed_df.to_csv(output_file, index=False)

print("Processed data saved to:", output_file)
print("Processed dataset shape:", processed_df.shape)

Processed data saved to: C:\Users\steph\Desktop\Willis College Courses\Fundamental of Predictive Analysis ML\Final Project\data\processed\cleaned_tweets.csv
Processed dataset shape: (1577939, 2)


# Model Training

### - Load the cleaned data and create a validation split

In [18]:
# Load the cleaned tweets produced in the preprocessing part
model_df = pd.read_csv(Path.cwd() / "data" / "processed" / "cleaned_tweets.csv")

In [19]:
# Convert the dataset labels: 0 = negative, 4 = positive
model_df["sentiment"] = model_df["sentiment"].map({0: 0, 4: 1})

In [20]:
# Separate tweet text from labels
X = model_df["clean_text"].astype(str)
y = model_df["sentiment"]

In [21]:
# Keep 20% as a separate validation dataset
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))

Training rows: 1262351
Validation rows: 315588


### - Convert text to numbers and train Logistic Regression

In [22]:
# Convert tweets into TF-IDF features
vectorizer = TfidfVectorizer(max_features=20_000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

In [23]:
# Train the Logistic Regression sentiment model
model = LogisticRegression(max_iter=200)
model.fit(X_train_tfidf, y_train)

print("Model training is complete.")

Model training is complete.


### - Validate using accuracy and F1-score

In [24]:
# Predict sentiments for the separate validation dataset
y_pred = model.predict(X_val_tfidf)

In [25]:
# Calculate the requested validation metrics
accuracy = accuracy_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)

print(f"Validation accuracy: {accuracy:.4f}")
print(f"Validation F1-score: {f1:.4f}")

Validation accuracy: 0.7966
Validation F1-score: 0.7990


# API Development and Containerization

### - Save the trained model and vectorizer

In [30]:
# Save the trained model and TF-IDF vectorizer for the Flask API
models_dir = Path.cwd() / "models"
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model, models_dir / "sentiment_model.joblib")
joblib.dump(vectorizer, models_dir / "tfidf_vectorizer.joblib")

# Record these versions for the Docker dependencies
print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)
print("Saved both model files in:", models_dir)

scikit-learn: 1.8.0
joblib: 1.5.3
Saved both model files in: C:\Users\steph\Desktop\Willis College Courses\Fundamental of Predictive Analysis ML\Final Project\models


### - app.py

### - requirements.txt

### - Dockerfile

### - .dockerignore

# MLOps Integration